In [0]:
import os, sys
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), "src"))
from lh import common, bronze

cfg = common.load_config("dev")
ok = True
print(f"{'table':20} {'rows':>6} {'files':>5} {'no_date':>7} {'rescued':>7} {'fingerprint':>11}  missing")
for table, spec in bronze.load_tables().items():
    df = spark.table(f"{cfg['catalog']}.bronze.{table}")
    need = {"_source_file", "_business_date", "_ingest_ts"}
    if spec["format"] != "binary":
        need.add("_rescued_data")
    if spec.get("fingerprint"):
        need.add("id_fingerprint")
    missing = sorted(need - set(df.columns))
    rows = df.count()
    files = df.select("_source_file").distinct().count()
    no_date = df.filter("_business_date IS NULL").count()
    rescued = df.filter("_rescued_data IS NOT NULL").count() if "_rescued_data" in df.columns else 0
    fp = ""
    if spec.get("fingerprint") and not missing:
        fp = f"{100 * df.filter('id_fingerprint IS NOT NULL').count() / rows:.1f}%"
    if missing or no_date or rows == 0:
        ok = False
    print(f"{table:20} {rows:>6} {files:>5} {no_date:>7} {rescued:>7} {fp:>11}  {missing or ''}")
print("PASS" if ok else "FAIL")